# Phase 5 & 6 — Big Data Analytics

## Why Big Data Frameworks?

Pandas is limited to a single machine's RAM. Real banks process **millions of transactions daily**.

| Approach | Limit | Tool |
|----------|-------|------|
| Single machine | ~50GB RAM | Pandas |
| Cluster (10 nodes) | Unlimited | PySpark + HDFS |

**Example:** VISA processes ~65,000 transactions per second globally.

At that scale, a single Pandas job would take hours — PySpark distributes work across hundreds of nodes in minutes.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

sys.path.append('../')
warnings.filterwarnings('ignore')
plt.style.use('dark_background')

print('Standard libraries loaded.')

## PySpark Setup

PySpark requires a running Java Virtual Machine (JVM). We use a try/except to gracefully handle environments where Spark is not installed.

In [ ]:
from src.spark_pipeline import create_spark_session, run_spark_analysis

try:
    spark = create_spark_session()
    print(f'Spark session created: {spark.version}')
    print(f'Master URL: {spark.sparkContext.master}')
    SPARK_AVAILABLE = True
except Exception as e:
    print(f'Spark not available: {e}')
    print('Running in SIMULATION MODE — demonstrating Spark API concepts.')
    print('To run with real Spark: pip install pyspark && java -version')
    SPARK_AVAILABLE = False
    spark = None

## Load Data into Spark DataFrame

In production: data lives in HDFS. For demo, we load from local CSV.

**HDFS path example:** `hdfs://namenode:9000/user/fraud/creditcard.csv`

In [ ]:
if SPARK_AVAILABLE:
    # Load CSV into Spark DataFrame
    sdf = spark.read.csv('../data/creditcard.csv', header=True, inferSchema=True)

    print('Spark DataFrame Schema:')
    sdf.printSchema()

    print(f'\nTotal records: {sdf.count():,}')

    print('\nClass distribution (Spark groupBy):')
    sdf.groupBy('Class').count().orderBy('Class').show()

    # Cache for repeated queries
    sdf.cache()
    print('DataFrame cached in Spark memory.')

else:
    print('=== SIMULATION MODE ===')
    print('Equivalent Spark code:')
    print('''
    sdf = spark.read.csv('hdfs:///user/fraud/creditcard.csv', header=True, inferSchema=True)
    sdf.printSchema()
    print(f"Total records: {sdf.count():,}")
    sdf.groupBy('Class').count().orderBy('Class').show()
    sdf.cache()
    ''')

    # Show equivalent Pandas output for demo
    df = pd.read_csv('../data/creditcard.csv')
    print('Pandas equivalent output:')
    print(df.groupby('Class').size().reset_index(name='count'))

## Spark MLlib Pipeline

Spark ML Pipelines encapsulate preprocessing + model in one object — no data leakage.

**Pipeline stages:**
1. `VectorAssembler` — combine feature columns into one vector column
2. `StandardScaler` — normalize features
3. `RandomForestClassifier` — train classifier

The entire pipeline is serializable — save to HDFS and reload on any cluster node.

In [ ]:
from src.spark_pipeline import build_spark_pipeline

feature_cols = [f'V{i}' for i in range(1, 29)] + ['Amount', 'Time']

if SPARK_AVAILABLE:
    print('Building Spark ML Pipeline...')
    pipeline = build_spark_pipeline(feature_cols)

    # Split Spark DataFrame
    train_sdf, test_sdf = sdf.randomSplit([0.8, 0.2], seed=42)

    # Fit pipeline
    print('Fitting pipeline on training data...')
    pipeline_model = pipeline.fit(train_sdf)

    # Transform test set
    predictions = pipeline_model.transform(test_sdf)

    # Evaluate
    from pyspark.ml.evaluation import BinaryClassificationEvaluator
    evaluator = BinaryClassificationEvaluator(labelCol='Class', metricName='areaUnderROC')
    roc_auc = evaluator.evaluate(predictions)
    print(f'\nSpark MLlib Random Forest ROC-AUC: {roc_auc:.4f}')

else:
    print('=== SIMULATION MODE ===')
    print('Equivalent Spark MLlib code:')
    print('''
    from pyspark.ml import Pipeline
    from pyspark.ml.feature import VectorAssembler, StandardScaler
    from pyspark.ml.classification import RandomForestClassifier
    from pyspark.ml.evaluation import BinaryClassificationEvaluator

    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")
    scaler = StandardScaler(inputCol="features_raw", outputCol="features")
    rf = RandomForestClassifier(labelCol="Class", numTrees=100)

    pipeline = Pipeline(stages=[assembler, scaler, rf])
    model = pipeline.fit(train_sdf)
    predictions = model.transform(test_sdf)

    evaluator = BinaryClassificationEvaluator(labelCol="Class", metricName="areaUnderROC")
    print(f"ROC-AUC: {evaluator.evaluate(predictions):.4f}")
    ''')

## Hadoop HDFS Integration

HDFS (Hadoop Distributed File System) stores data across multiple nodes for fault tolerance.

**Key concepts:**
- **NameNode:** Stores metadata (directory structure, file locations)
- **DataNode:** Stores actual data blocks (default 128MB per block)
- **Replication factor:** Each block stored on 3 nodes by default
- **Fault tolerance:** If one node dies, data is safe on other 2 nodes

In [ ]:
# HDFS shell commands (run from terminal or using subprocess)
# These demonstrate the HDFS workflow for this project

hdfs_commands = {
    'Create project directory': 'hdfs dfs -mkdir -p /user/fraud/data',
    'Upload dataset': 'hdfs dfs -put /local/path/creditcard.csv /user/fraud/data/',
    'List files': 'hdfs dfs -ls /user/fraud/data/',
    'Check file size': 'hdfs dfs -du -h /user/fraud/data/',
    'View first 10 lines': 'hdfs dfs -cat /user/fraud/data/creditcard.csv | head -10',
    'Copy to local': 'hdfs dfs -get /user/fraud/data/creditcard.csv /local/download/',
    'Show disk usage': 'hdfs dfsadmin -report',
    'Check replication': 'hdfs fsck /user/fraud/data/creditcard.csv -files -blocks'
}

print('HDFS Commands for this project:')
print('=' * 60)
for description, cmd in hdfs_commands.items():
    print(f'\n# {description}')
    print(f'$ {cmd}')

print('\n' + '=' * 60)
print('In Spark, read directly from HDFS:')
print('  sdf = spark.read.csv("hdfs://namenode:9000/user/fraud/data/creditcard.csv")')

## MapReduce Fraud Analysis

MapReduce is the original Hadoop computation model — still useful for simple aggregations.

In [ ]:
from src.mapreduce_fraud import run_fraud_mapreduce

print('Running MapReduce fraud analysis...')
results = run_fraud_mapreduce('../data/creditcard.csv')

print('\nMapReduce Results:')
print('=' * 50)
for key, value in results.items():
    print(f'{key}: {value}')

## MapReduce Explanation

**Map Phase:** Each transaction → `(class, amount)`
```
Transaction(V1..V28, Amount=120, Class=0) → ('legit', 120.0)
Transaction(V1..V28, Amount=50,  Class=1) → ('fraud', 50.0)
```

**Shuffle Phase:** Group by class key
```
'legit' → [120.0, 88.0, 45.0, ...]
'fraud' → [50.0, 22.0, 115.0, ...]
```

**Reduce Phase:** Sum amounts, count per class
```
'legit' → {count: 284315, total_amount: 25591953.0, avg: 90.0}
'fraud' → {count: 492, total_amount: 10790.0, avg: 22.0}
```

This scales to **billions of records** across a cluster — each node processes its local data shard in parallel, then results are merged.